# 00 · Generator Dataset Latihan
**Pelatihan Machine Learning & Data Analysis**

Notebook ini membuat **3 dataset sintetis** (data buatan, bukan data asli perusahaan) yang dipakai di seluruh latihan:

| Dataset | Dipakai di | Isi |
|---|---|---|
| `data_penjualan_motor.csv` | Modul 2 & 3 | 53 bulan riwayat penjualan (tren + musiman + efek promo) |
| `data_transaksi_sparepart.csv` | Modul 1 & 4 | 1.218 transaksi, sengaja dibuat "kotor" + 12 anomali tersembunyi |
| `data_kredit_konsumen.csv` | Modul 3 & 5 | 1.500 profil konsumen kredit (Lancar / Macet) |

> ▶️ **Jalankan notebook ini SEKALI di awal**, sebelum membuka Modul 1–5.
> Cara pakai: jalankan sel dari atas ke bawah dengan `Shift+Enter`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Supaya hasil acak selalu sama setiap kali dijalankan (reproducible)
rng = np.random.default_rng(42)

# Folder penyimpanan dataset (relatif terhadap folder notebook)
DIR_DATASET = Path.cwd().parent / "dataset"
DIR_DATASET.mkdir(exist_ok=True)

## 1. DATA PENJUALAN MOTOR BULANAN (Jan 2022 - Mei 2026)
*Punya pola: tren naik + musiman (ramai jelang Lebaran & akhir tahun)*

In [ ]:
print("Membuat data penjualan motor bulanan ...")

bulan = pd.date_range("2022-01-01", "2026-05-01", freq="MS")  # MS = awal bulan
n = len(bulan)

tren = np.linspace(11000, 16500, n)                      # tren naik bertahap
musiman = 1600 * np.sin(2 * np.pi * (bulan.month - 3) / 12)  # pola musiman
promo = rng.choice([0, 1], size=n, p=[0.7, 0.3])         # 30% bulan ada promo
efek_promo = promo * rng.normal(1200, 200, n)            # promo menaikkan penjualan
acak = rng.normal(0, 450, n)                             # variasi acak (noise)

unit_terjual = (tren + musiman + efek_promo + acak).round().astype(int)

df_penjualan = pd.DataFrame({
    "bulan": bulan.strftime("%Y-%m"),
    "tahun": bulan.year,
    "no_bulan": bulan.month,
    "ada_promo": promo,
    "unit_terjual": unit_terjual,
})
df_penjualan.to_csv(DIR_DATASET / "data_penjualan_motor.csv", index=False)
print(f"  -> {len(df_penjualan)} baris : data_penjualan_motor.csv")

## 2. DATA TRANSAKSI SPAREPART (sengaja dibuat "kotor" + ada anomali)
*Dipakai untuk latihan pembersihan data (modul 1) dan deteksi anomali (modul 4)*

In [ ]:
print("Membuat data transaksi sparepart ...")

n_trx = 1200
nama_part = ["Oli Mesin", "Kampas Rem", "Filter Udara", "Busi", "Ban Luar", "Aki"]
harga_normal = {"Oli Mesin": 55000, "Kampas Rem": 75000, "Filter Udara": 60000,
                "Busi": 35000, "Ban Luar": 250000, "Aki": 280000}
dealer = [f"Dealer-{k}" for k in ["JKT01", "JKT02", "BDG01", "SBY01", "MDN01", "SMG01"]]

tanggal = pd.to_datetime("2026-01-01") + pd.to_timedelta(
    rng.integers(0, 150, n_trx), unit="D")
part = rng.choice(nama_part, n_trx)
qty = rng.integers(1, 8, n_trx)
harga = np.array([harga_normal[p] for p in part], dtype=float)
harga = harga * rng.normal(1.0, 0.03, n_trx)  # variasi harga wajar +-3%

df_trx = pd.DataFrame({
    "id_transaksi": [f"TRX{100000 + i}" for i in range(n_trx)],
    "tanggal": tanggal.strftime("%Y-%m-%d"),
    "dealer": rng.choice(dealer, n_trx),
    "nama_part": part,
    "qty": qty,
    "harga_satuan": harga.round(-2),  # dibulatkan ke ratusan
})
df_trx["total"] = df_trx["qty"] * df_trx["harga_satuan"]

### Suntik ANOMALI (kejadian tidak wajar yang harus ditemukan AI)

In [ ]:
idx_anomali = rng.choice(n_trx, 12, replace=False)
for i, idx in enumerate(idx_anomali):
    if i < 5:    # qty tidak wajar (misal salah input / pembelian mencurigakan)
        df_trx.loc[idx, "qty"] = int(rng.integers(60, 150))
    elif i < 9:  # harga jauh di atas normal (markup mencurigakan)
        df_trx.loc[idx, "harga_satuan"] = df_trx.loc[idx, "harga_satuan"] * 12
    else:        # harga hampir nol (potensi fraud/diskon ilegal)
        df_trx.loc[idx, "harga_satuan"] = 1000
df_trx["total"] = df_trx["qty"] * df_trx["harga_satuan"]

### Kotori data (untuk latihan pembersihan di modul 1)

In [ ]:
kotor = df_trx.copy()
idx_kosong = rng.choice(n_trx, 25, replace=False)
kotor.loc[idx_kosong[:15], "qty"] = np.nan          # qty kosong
kotor.loc[idx_kosong[15:], "dealer"] = None         # dealer kosong
idx_teks = rng.choice(n_trx, 30, replace=False)
kotor.loc[idx_teks, "nama_part"] = (                # penulisan tidak konsisten
    kotor.loc[idx_teks, "nama_part"].str.upper().str.replace(" ", "_"))
duplikat = kotor.sample(18, random_state=42)        # baris duplikat
kotor = pd.concat([kotor, duplikat], ignore_index=True)
kotor = kotor.sample(frac=1, random_state=42).reset_index(drop=True)  # acak urutan

kotor.to_csv(DIR_DATASET / "data_transaksi_sparepart.csv", index=False)
print(f"  -> {len(kotor)} baris : data_transaksi_sparepart.csv "
      f"(berisi data kosong, duplikat, & {len(idx_anomali)} anomali)")

## 3. DATA KREDIT KONSUMEN (untuk latihan klasifikasi: Lancar vs Macet)

In [ ]:
print("Membuat data kredit konsumen ...")

n_kredit = 1500
usia = rng.integers(19, 56, n_kredit)
penghasilan = rng.normal(7, 2.5, n_kredit).clip(2.5, 20).round(1)   # juta/bulan
dp_persen = rng.choice([10, 15, 20, 25, 30], n_kredit, p=[.30, .25, .20, .15, .10])
tenor = rng.choice([12, 24, 36], n_kredit, p=[.25, .45, .30])
telat_sebelumnya = rng.poisson(0.8, n_kredit).clip(0, 6)

# Skor risiko: makin tinggi -> makin mungkin macet
skor = (
    -0.40 * (penghasilan - 7)
    - 0.13 * (dp_persen - 18)
    + 0.95 * telat_sebelumnya
    + 0.05 * (tenor / 12)
    - 0.02 * (usia - 35)
    + rng.normal(0, 0.6, n_kredit)   # faktor tak terukur (noise wajar)
)
status = np.where(skor > 1.7, "Macet", "Lancar")

df_kredit = pd.DataFrame({
    "id_konsumen": [f"K{20000 + i}" for i in range(n_kredit)],
    "usia": usia,
    "penghasilan_juta": penghasilan,
    "dp_persen": dp_persen,
    "tenor_bulan": tenor,
    "jumlah_telat_sebelumnya": telat_sebelumnya,
    "status_pembayaran": status,
})
df_kredit.to_csv(DIR_DATASET / "data_kredit_konsumen.csv", index=False)
print(f"  -> {len(df_kredit)} baris : data_kredit_konsumen.csv "
      f"({(status == 'Macet').mean():.0%} macet)")

print("\nSelesai! Semua dataset tersimpan di folder 'dataset/'.")

---
✅ **Selesai.** Tiga dataset sudah tersimpan di folder `dataset/`. Lanjut ke `01_pengolahan_data.ipynb`.